In [ ]:
import os
import sys
import pandas as pd

# Move up one directory to the project root and add it to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports will work perfectly!
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.eda_utils import get_missing_summary, plot_loss_ratio_by_dimension

In [ ]:
# Execution code within notebook context

from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.hypothesis_tests import run_categorical_chi2_test, run_numerical_ttest

# Point directly to your zipped text file path
zip_path = '../data/MachineLearningRating_v3.zip' 

df = load_insurance_data(zip_path)
df = calculate_insurance_metrics(df)

results_summary = []

# Example Test 1: Gender vs Claim Frequency
chi2_g, p_g = run_categorical_chi2_test(df, 'Gender', 'TotalClaims')
results_summary.append({
    'Hypothesis': 'Risk variance across Gender labels',
    'Test Type': 'Chi-Square Contingency',
    'p-value': p_g,
    'Decision': 'Reject H0' if p_g < 0.05 else 'Fail to Reject H0'
})

In [ ]:
# Example Test 2: Selected Province Pairs on Claim Severity (Only considering positive claims records)
claims_only = df[df['TotalClaims'] > 0]
if len(df['Province'].unique()) >= 2:
    p1, p2 = df['Province'].value_counts().index[0], df['Province'].value_counts().index[1]
    t_p, p_p = run_numerical_ttest(claims_only, 'Province', p1, p2, 'TotalClaims')
    results_summary.append({
        'Hypothesis': f'Claim Severity Variance ({p1} vs {p2})',
        'Test Type': 'Welch t-test',
        'p-value': p_p,
        'Decision': 'Reject H0' if p_p < 0.05 else 'Fail to Reject H0'
})

print(pd.DataFrame(results_summary).to_markdown(index=False))
